# 分布式推理 TP/PP 通信模拟 + Roofline 分析

本 notebook 包含：
1. **GPU 硬件配置 + 模型参数 dataclass**
2. **Roofline 模型**（区分 prefill compute-bound vs decode memory-bound）
3. **Tensor Parallel (TP) AllReduce 通信量分析**
4. **Pipeline Parallel (PP) Bubble 率计算**
5. **TP × PP 混合并行最优配置搜索**
6. **Decode 吞吐随 Batch Size 变化**
7. **面试可讲的结论模板**

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Dict, List

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

print(f'Ready. matplotlib: {HAS_MPL}')

## 1) GPU 硬件配置 + 模型参数

In [ ]:
@dataclass
class GPUConfig:
    """GPU 硬件参数。"""
    name: str = "H100-SXM"
    fp16_tflops: float = 990.0
    fp8_tflops: float = 1979.0
    hbm_bandwidth_tb_s: float = 3.35
    hbm_capacity_gb: float = 80.0
    nvlink_bw_gb_s: float = 900.0
    pcie_bw_gb_s: float = 128.0
    ib_bw_gb_s: float = 400.0

    @property
    def roofline_knee_fp16(self) -> float:
        """拐点 = TFLOPS / (TB/s) = FLOP/Byte"""
        return self.fp16_tflops / self.hbm_bandwidth_tb_s


@dataclass
class ModelConfig:
    """模型参数。"""
    name: str = "Llama3-70B"
    n_layers: int = 80
    d_model: int = 8192
    n_heads: int = 64
    n_kv_heads: int = 8
    head_dim: int = 128
    d_ff: int = 28672
    vocab_size: int = 128256
    dtype_bytes: int = 2

    @property
    def param_count(self) -> int:
        attn_per_layer = self.d_model * (self.n_heads + 2 * self.n_kv_heads) * self.head_dim + self.d_model * self.d_model
        ffn_per_layer = 3 * self.d_model * self.d_ff
        embed = self.vocab_size * self.d_model
        return self.n_layers * (attn_per_layer + ffn_per_layer) + 2 * embed

    @property
    def weight_bytes(self) -> int:
        return self.param_count * self.dtype_bytes

    def kv_bytes_per_token(self) -> int:
        return 2 * self.n_layers * self.n_kv_heads * self.head_dim * self.dtype_bytes


gpu = GPUConfig()
model = ModelConfig()

print(f'GPU: {gpu.name}')
print(f'  FP16 TFLOPS: {gpu.fp16_tflops}')
print(f'  HBM BW: {gpu.hbm_bandwidth_tb_s} TB/s')
print(f'  Roofline knee: {gpu.roofline_knee_fp16:.1f} FLOP/Byte')
print(f'Model: {model.name}')
print(f'  Params: {model.param_count/1e9:.1f}B')
print(f'  Weight: {model.weight_bytes/1e9:.1f} GB')
print(f'  KV bytes/token: {model.kv_bytes_per_token()/1024:.1f} KB')

## 2) Roofline 模型：Prefill vs Decode

In [ ]:
def prefill_flops_per_token(m: ModelConfig) -> float:
    """Prefill 每 token 的 FLOPs（matmul 主导）。"""
    attn_flops = 2 * m.d_model * (m.n_heads + 2 * m.n_kv_heads) * m.head_dim
    attn_flops += 2 * m.d_model * m.d_model
    ffn_flops = 3 * 2 * m.d_model * m.d_ff
    return (attn_flops + ffn_flops) * m.n_layers


def decode_flops_per_step(m: ModelConfig, seq_len: int) -> float:
    """Decode 单步 FLOPs（1 token query against seq_len KV）。"""
    proj_flops = prefill_flops_per_token(m)
    attn_flops = 2 * m.n_heads * seq_len * m.head_dim * m.n_layers
    return proj_flops + attn_flops


def decode_bytes_per_step(m: ModelConfig, seq_len: int, batch_size: int = 1) -> float:
    """Decode 每步读取数据量（权重 + KV Cache）。"""
    return m.weight_bytes + batch_size * m.kv_bytes_per_token() * seq_len


def arithmetic_intensity(flops: float, bytes_accessed: float) -> float:
    return flops / bytes_accessed if bytes_accessed > 0 else float('inf')


def roofline_throughput(ai: float, g: GPUConfig) -> float:
    return min(g.fp16_tflops, g.hbm_bandwidth_tb_s * ai)


# Prefill 分析
pf = prefill_flops_per_token(model)
prefill_ai = arithmetic_intensity(pf * 4096, model.weight_bytes)

# Decode 分析 (batch=1, seq=4096)
df = decode_flops_per_step(model, 4096)
db = decode_bytes_per_step(model, 4096, 1)
decode_ai = arithmetic_intensity(df, db)

print(f'Prefill (T=4096):')
print(f'  FLOPs/token: {pf/1e9:.1f} GFLOPs')
print(f'  Arithmetic Intensity (batch): {prefill_ai:.1f} FLOP/Byte')
print(f'  Regime: {"compute-bound" if prefill_ai > gpu.roofline_knee_fp16 else "memory-bound"}')

print(f'\nDecode (T=4096, batch=1):')
print(f'  FLOPs/step: {df/1e9:.1f} GFLOPs')
print(f'  Bytes/step: {db/1e9:.1f} GB')
print(f'  Arithmetic Intensity: {decode_ai:.2f} FLOP/Byte')
print(f'  Regime: {"compute-bound" if decode_ai > gpu.roofline_knee_fp16 else "memory-bound"}')
print(f'\nRoofline knee: {gpu.roofline_knee_fp16:.1f} FLOP/Byte')

## 3) Roofline 图

if HAS_MPL:
    ai_range = np.logspace(-1, 4, 500)
    throughput = np.minimum(gpu.fp16_tflops, gpu.hbm_bandwidth_tb_s * ai_range)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.loglog(ai_range, throughput, 'b-', lw=2, label='Roofline (H100 FP16)')
    ax.axvline(gpu.roofline_knee_fp16, color='gray', ls='--', alpha=0.5,
               label=f'Knee = {gpu.roofline_knee_fp16:.0f} FLOP/B')

    for batch in [1, 8, 32, 128]:
        d_f = decode_flops_per_step(model, 4096) * batch
        d_b = decode_bytes_per_step(model, 4096, batch)
        d_ai = arithmetic_intensity(d_f, d_b)
        d_tp = roofline_throughput(d_ai, gpu)
        ax.plot(d_ai, d_tp, 'ro', markersize=8)
        ax.annotate(f'decode B={batch}', (d_ai, d_tp),
                   textcoords='offset points', xytext=(10, 5), fontsize=8)

    for T in [512, 2048, 8192]:
        p_f = prefill_flops_per_token(model) * T
        p_b = model.weight_bytes
        p_ai = arithmetic_intensity(p_f, p_b)
        p_tp = roofline_throughput(p_ai, gpu)
        ax.plot(p_ai, p_tp, 'g^', markersize=8)
        ax.annotate(f'prefill T={T}', (p_ai, p_tp),
                   textcoords='offset points', xytext=(10, -10), fontsize=8)

    ax.set_xlabel('Arithmetic Intensity (FLOP/Byte)', fontsize=12)
    ax.set_ylabel('Throughput (TFLOPS)', fontsize=12)
    ax.set_title(f'Roofline Model: {gpu.name} + {model.name}', fontsize=14)
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('matplotlib 不可用，跳过 Roofline 图')
    print(f'Decode AI={decode_ai:.2f}, Prefill AI={prefill_ai:.1f}, Knee={gpu.roofline_knee_fp16:.1f}')

## 4) Tensor Parallel 通信量分析

In [ ]:
def tp_allreduce_bytes_per_layer(m: ModelConfig, batch_size: int, seq_len: int) -> float:
    """TP 每层 AllReduce 通信量（2 次 AllReduce：attn + FFN）。"""
    single = batch_size * seq_len * m.d_model * m.dtype_bytes
    return 2 * single * 2  # 2 AllReduce × ring ~2× data


def tp_total_comm_time(m: ModelConfig, g: GPUConfig, tp_degree: int,
                       batch_size: int, seq_len: int) -> Dict[str, float]:
    per_layer = tp_allreduce_bytes_per_layer(m, batch_size, seq_len)
    total = per_layer * m.n_layers
    bw = g.nvlink_bw_gb_s if tp_degree <= 8 else g.ib_bw_gb_s
    comm_s = total / (bw * 1e9)
    return {
        "per_layer_mb": per_layer / 1e6,
        "total_gb": total / 1e9,
        "bw_gb_s": bw,
        "comm_ms": comm_s * 1000,
    }


print(f'TP 通信量分析 ({model.name})')
print(f'{"TP":>4} {"batch":>6} {"seq":>6} {"total_GB":>10} {"comm_ms":>10} {"link":>10}')
for tp in [2, 4, 8]:
    for bs, sl in [(1, 1), (1, 4096), (32, 4096)]:
        r = tp_total_comm_time(model, gpu, tp, bs, sl)
        link = 'NVLink' if tp <= 8 else 'IB'
        print(f'{tp:>4} {bs:>6} {sl:>6} {r["total_gb"]:>10.3f} {r["comm_ms"]:>10.2f} {link:>10}')

## 5) Pipeline Parallel Bubble 率分析

In [ ]:
def pp_bubble_ratio(pp_degree: int, num_micro_batches: int) -> float:
    if num_micro_batches <= 0:
        return 1.0
    return (pp_degree - 1) / num_micro_batches


def pp_efficiency(pp_degree: int, num_micro_batches: int) -> float:
    return 1.0 - pp_bubble_ratio(pp_degree, num_micro_batches)


def pp_activation_bytes(m: ModelConfig, batch_size: int, seq_len: int) -> float:
    return batch_size * seq_len * m.d_model * m.dtype_bytes


print('PP Bubble 率分析')
print(f'{"PP":>6} {"micro_batches":>14} {"bubble":>8} {"efficiency":>12}')
for pp in [2, 4, 8]:
    for mb in [4, 8, 16, 32, 64]:
        b = pp_bubble_ratio(pp, mb)
        e = pp_efficiency(pp, mb)
        print(f'{pp:>6} {mb:>14} {b:>8.1%} {e:>12.1%}')

print(f'\nPP 激活传输量 (per micro-batch, seq=4096, batch=1): '
      f'{pp_activation_bytes(model, 1, 4096)/1e6:.1f} MB')

## 6) TP × PP 混合并行最优配置搜索

In [ ]:
def evaluate_config(m: ModelConfig, g: GPUConfig, total_gpus: int,
                    tp: int, pp: int, batch_size: int = 32,
                    seq_len: int = 4096, num_micro_batches: int = 16) -> dict:
    if tp * pp > total_gpus or m.n_layers % pp != 0:
        return None
    w_per_gpu = m.weight_bytes / (tp * pp) / 1e9
    kv_per_gpu = (m.kv_bytes_per_token() / tp * seq_len * batch_size / pp) / 1e9
    total_gpu = w_per_gpu + kv_per_gpu
    fits = total_gpu < g.hbm_capacity_gb * 0.9

    tp_comm = tp_total_comm_time(m, g, tp, batch_size // max(pp, 1), seq_len)
    pp_eff = pp_efficiency(pp, num_micro_batches)
    score = pp_eff / (1 + tp_comm['comm_ms'] / 100) if fits else 0

    return {
        'tp': tp, 'pp': pp, 'gpus': tp * pp,
        'w_gpu': round(w_per_gpu, 1), 'kv_gpu': round(kv_per_gpu, 1),
        'tot_gpu': round(total_gpu, 1), 'fits': fits,
        'tp_ms': round(tp_comm['comm_ms'], 2),
        'pp_eff': round(pp_eff, 3), 'score': round(score, 4),
    }


total_gpus = 16
print(f'=== TP×PP Config Search ({model.name}, {total_gpus} GPUs) ===')
print(f'{"TP":>4} {"PP":>4} {"GPUs":>5} {"W/GPU":>7} {"KV/GPU":>7} {"Tot":>6} '
      f'{"Fits":>5} {"TP_ms":>7} {"PP_eff":>7} {"Score":>7}')

results = []
for tp in [1, 2, 4, 8]:
    for pp in [1, 2, 4, 8, 16]:
        r = evaluate_config(model, gpu, total_gpus, tp, pp)
        if r is None:
            continue
        results.append(r)
        print(f'{r["tp"]:>4} {r["pp"]:>4} {r["gpus"]:>5} {r["w_gpu"]:>7} {r["kv_gpu"]:>7} '
              f'{r["tot_gpu"]:>6} {str(r["fits"]):>5} {r["tp_ms"]:>7} {r["pp_eff"]:>7} {r["score"]:>7}')

best = max(results, key=lambda x: x['score'])
print(f'\n✅ Best: TP={best["tp"]}, PP={best["pp"]} (score={best["score"]})')

## 7) Decode 吞吐随 Batch Size 变化

In [ ]:
def decode_throughput_tok_s(m: ModelConfig, g: GPUConfig,
                           batch_size: int, seq_len: int, tp: int = 1) -> float:
    flops = decode_flops_per_step(m, seq_len) * batch_size / tp
    bts = decode_bytes_per_step(m, seq_len, batch_size) / tp
    ai = arithmetic_intensity(flops, bts)
    achieved = roofline_throughput(ai, g)
    time_s = flops / (achieved * 1e12) if achieved > 0 else float('inf')
    return batch_size / time_s


batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]
print(f'{"Batch":>6} {"AI":>8} {"tok/s":>10} {"TPOT(ms)":>10}')
throughputs = []
for bs in batch_sizes:
    tps = decode_throughput_tok_s(model, gpu, bs, 4096, tp=8)
    throughputs.append(tps)
    tpot = 1000 / tps * bs if tps > 0 else float('inf')
    d_f = decode_flops_per_step(model, 4096) * bs / 8
    d_b = decode_bytes_per_step(model, 4096, bs) / 8
    ai = arithmetic_intensity(d_f, d_b)
    print(f'{bs:>6} {ai:>8.2f} {tps:>10.0f} {tpot:>10.2f}')

In [ ]:
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(batch_sizes, throughputs, 'bo-', lw=2)
    ax1.set_xlabel('Batch Size', fontsize=12)
    ax1.set_ylabel('Decode Throughput (tok/s)', fontsize=12)
    ax1.set_title(f'Decode Throughput vs Batch ({model.name}, TP=8)', fontsize=13)
    ax1.set_xscale('log', base=2); ax1.grid(True, alpha=0.3)

    for pp in [2, 4, 8]:
        mbs = list(range(pp, 65))
        effs = [pp_efficiency(pp, m) for m in mbs]
        ax2.plot(mbs, effs, label=f'PP={pp}')
    ax2.set_xlabel('Number of Micro-batches', fontsize=12)
    ax2.set_ylabel('Pipeline Efficiency', fontsize=12)
    ax2.set_title('PP Efficiency vs Micro-batches', fontsize=13)
    ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3); ax2.set_ylim(0, 1.05)

    plt.tight_layout(); plt.show()
else:
    print('matplotlib 不可用，跳过可视化')

## 8) 面试可讲的结论模板

> **Roofline 分析**：
> "Prefill 是 compute-bound（AI >> knee），Decode 是 memory-bound（AI << knee）。
> 提高 decode 吞吐的关键是增大 batch size 来提升算术强度。"

> **TP vs PP 选择**：
> "机内用 TP（NVLink 900 GB/s），跨机用 PP（IB 400 GB/s）。
> TP 的通信量随 batch×seq 增长，PP 的瓶颈是 bubble（需要足够 micro-batch）。"

> **70B 模型推荐配置**：
> "TP=8（单机 8 卡 H100），如果 1 台不够就 TP=8 × PP=2/4。
> PP bubble 率控制在 <10%（micro-batch >= 8×PP）。"

### 面试追问
1. **TP 通信量怎么算？** → 每层 2 次 AllReduce，每次 B×T×d×dtype bytes
2. **PP bubble 怎么减？** → 增加 micro-batch 数；或用 1F1B 调度
3. **为什么不全用 TP？** → TP 度 > 8 需跨机通信（IB 带宽远低于 NVLink），延迟大增
4. **EP 什么时候用？** → MoE 模型，expert 太多放不下时用 EP 分摊